In [5]:
import csv
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Tuple
import os
from pathlib import Path

# import your inflection helpers
from grammar import apply_tags

# ---------- basic Levenshtein (edit distance) ----------
def _levenshtein(a: str, b: str) -> int:
    a, b = a.lower(), b.lower()
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            cur.append(min(
                prev[j] + 1,      # deletion
                cur[j-1] + 1,     # insertion
                prev[j-1] + cost  # substitution
            ))
        prev = cur
    return prev[-1]

def _too_close(x: str, y: str, min_distance: int = 2) -> bool:
    # “less than 2” means distance 0 or 1 is too close
    return _levenshtein(x, y) < min_distance

# ---------- parsing placeholders ----------
# Matches: [noun_1][plural][capitalize], [verb_2][past], [rel_1] etc.
PH_RE = re.compile(
    r"(?P<prefix>(?:\[negate\])*)"                               # optional [negate]
    r"\[(?P<kind>noun|verb|adj|rel)_(?P<idx>\d+)\]"              # slot
    r"(?P<suffix>(?:\[(?:capitalize|plural|past|progressive|negate|abbr)\])*)"  # tags
)


def _extract_placeholders(template: str):
    """
    Return a list of (kind:str, idx:int, tags:List[str]) for each placeholder occurrence.
    Tags may include: negate, capitalize, plural, past, progressive.
    """
    results = []
    for m in PH_RE.finditer(template):
        kind = m.group("kind")
        idx = int(m.group("idx"))

        # collect tags from both prefix and suffix
        tags = []
        if m.group("prefix"):
            tags += re.findall(r"\[(negate)\]", m.group("prefix"))
        if m.group("suffix"):
            tags += re.findall(r"\[(capitalize|plural|past|progressive|negate|abbr)\]", m.group("suffix"))

        results.append((kind, idx, tags))
    return results

def _indices_by_kind(template: str):
    """
    Collect unique indices required per kind.
    """
    placeholders = _extract_placeholders(template)
    by_kind = {"noun": set(), "verb": set(), "adj": set(), "rel": set()}
    for kind, idx, _ in placeholders:
        by_kind[kind].add(idx)
    return {k: sorted(v) for k, v in by_kind.items()}

# ---------- selection with uniqueness + distance constraints ----------
def _choose_unique(words: List[str], disallow: List[str]) -> str:
    """
    Choose a word not equal to any in disallow and with edit distance >= 2 to each.
    """
    candidates = words[:]
    random.shuffle(candidates)
    for w in candidates:
        if all((w.lower() != d.lower()) and (not _too_close(w, d)) for d in disallow):
            return w
    raise ValueError("No available candidate satisfies uniqueness/distance constraints.")

def _build_assignment(template: str,
                      nouns: List[str],
                      verbs: List[str],
                      adjs: List[str],
                      rels: List[str]) -> Dict[str, Dict[int, str]]:
    """
    Returns mapping like {'noun': {1:'foo',2:'bar'}, 'verb': {1:'...'}, ...}
    ensuring no reuse within a prompt and edit distance >= 2 across all chosen items.
    """
    needed = _indices_by_kind(template)
    chosen: Dict[str, Dict[int, str]] = {"noun": {}, "verb": {}, "adj": {}, "rel": {}}
    used: List[str] = []

    pools = {
        "noun": nouns[:],
        "verb": verbs[:],
        "adj":  adjs[:],
        "rel":  rels[:],
    }

    # ensure we don't pick the same surface form across lists
    # (we also rely on distance constraint below)
    for kind in ("noun", "verb", "adj", "rel"):
        random.shuffle(pools[kind])

    for kind in ("noun", "verb", "adj", "rel"):
        for idx in needed[kind]:
            choice = _choose_unique(pools[kind], used)
            chosen[kind][idx] = choice
            used.append(choice)

    return chosen

# ---------- filling ----------
def _replacer_factory(assignments):
    """
    Build a replacement function for re.sub that applies tags with grammar.apply_tags.
    """
    def repl(m: re.Match) -> str:
        kind = m.group("kind")
        idx = int(m.group("idx"))

        # Gather tags from both sides exactly as in _extract_placeholders
        tags = []
        if m.group("prefix"):
            tags += re.findall(r"\[(negate)\]", m.group("prefix"))
        if m.group("suffix"):
            tags += re.findall(r"\[(capitalize|plural|past|progressive|negate|abbr)\]", m.group("suffix"))

        base = assignments[kind][idx]

        # Relations: ignore everything except 'capitalize'
        if kind == "rel":
            eff_tags = [t for t in tags if t == "capitalize"]
        else:
            eff_tags = tags

        return apply_tags(base, eff_tags)
    return repl

# ---------- main driver ----------
def create_prompts(template_path: str,
                   noun_path: str,
                   verb_path: str,
                   adj_path: str,
                   rel_path: str,
                   out_csv_path: str,
                   n_samples: int = 20,
                   seed: int = None) -> None:
    """
    Generate n_samples filled prompts with randomized, mutually distinct words
    (and edit distance >=2 between any two chosen items per prompt), then save to CSV.

    CSV columns:
      id, mapping (JSON), prompt
    """
    if seed is not None:
        random.seed(seed)

    template = Path(template_path).read_text(encoding="utf-8")

    nouns = [w.strip() for w in Path(noun_path).read_text(encoding="utf-8").splitlines() if w.strip()]
    verbs = [w.strip() for w in Path(verb_path).read_text(encoding="utf-8").splitlines() if w.strip()]
    adjs  = [w.strip() for w in Path(adj_path).read_text(encoding="utf-8").splitlines() if w.strip()]
    rels  = [w.strip() for w in Path(rel_path).read_text(encoding="utf-8").splitlines() if w.strip()]

    # sanity check: enough unique items per category
    needs = _indices_by_kind(template)
    for k, pool in (("noun", nouns), ("verb", verbs), ("adj", adjs), ("rel", rels)):
        if len(set(pool)) < len(needs[k]):
            raise ValueError(f"Not enough unique {k}s: need {len(needs[k])}, have {len(set(pool))}.")

    rows = []
    seen_prompts = set()

    for i in range(n_samples):
        # Retry a few times to avoid duplicates due to randomness/collisions
        for attempt in range(50):
            assignments = _build_assignment(template, nouns, verbs, adjs, rels)
            filled = PH_RE.sub(_replacer_factory(assignments), template).strip()

            if filled not in seen_prompts:
                seen_prompts.add(filled)
                mapping = {
                    "noun": assignments["noun"],
                    "verb": assignments["verb"],
                    "adj": assignments["adj"],
                    "rel": assignments["rel"],
                }
                rows.append({
                    "id": i + 1,
                    "mapping": json.dumps(mapping, ensure_ascii=False),
                    "prompt": filled
                })
                break
        else:
            # No unique prompt found after many attempts; stop early
            print(f"Stopped early at {len(rows)} prompts — reached uniqueness limit.")
            break

    # write CSV
    with open(out_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "mapping", "prompt"])
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
import os
import csv
import json
import random
import re
from pathlib import Path
from typing import Dict, List

# --- import from your existing module ---
# from your_module import (
#     PH_RE,                 # regex that supports [negate] and [abbr]
#     _indices_by_kind,
#     _build_assignment,     # picks words with no-repeat + edit-distance >= 2
#     _replacer_factory,     # builds re.sub replacer calling grammar.apply_tags()
# )

# If your _replacer_factory currently *ignores* tags on relations except 'capitalize',
# update it to ALSO allow 'abbr' for relations (since PDDL keys need [abbr] on relations).
#
# Example tweak inside _replacer_factory:
#   if kind == "rel":
#       eff_tags = [t for t in tags if t in {"capitalize", "abbr"}]
#   else:
#       eff_tags = tags


def _fill_with_assignments(template_text: str, assignments: Dict[str, Dict[int, str]]) -> str:
    """Fill a single template with a fixed assignments dict."""
    return PH_RE.sub(_replacer_factory(assignments), template_text)

def _load_words(p: str) -> List[str]:
    return [w.strip() for w in Path(p).read_text(encoding="utf-8").splitlines() if w.strip()]

def _load_template(p: Path) -> str:
    return p.read_text(encoding="utf-8")

def create_prompts_triple(
    template_path: str,                 # original/natural template
    gen_domain_path: str,               # generated_domain template (same stem)
    pddl_path: str,                     # PDDL template (same stem)
    noun_path: str,
    verb_path: str,
    adj_path: str,
    rel_path: str,
    out_csv_path: str,
    n_samples: int = 20,
    seed: int | None = None,
) -> None:
    """Create a CSV where each row uses ONE assignment to fill 3 sibling templates."""
    if seed is not None:
        random.seed(seed)

    # Load templates
    t_orig = _load_template(Path(template_path))
    t_gend = _load_template(Path(gen_domain_path))
    t_pddl = _load_template(Path(pddl_path))

    # Load lexicons
    nouns = _load_words(noun_path)
    verbs = _load_words(verb_path)
    adjs  = _load_words(adj_path)
    rels  = _load_words(rel_path)

    # Sanity: ensure each template’s slot demands can be met
    # We compute for the UNION of all three templates, to guarantee one assignment fits all.
    def union_indices(*templates: List[str]) -> Dict[str, List[int]]:
        by = {"noun": set(), "verb": set(), "adj": set(), "rel": set()}
        for T in templates:
            need = _indices_by_kind(T)
            for k in by:
                by[k] |= set(need[k])
        return {k: sorted(v) for k, v in by.items()}

    needs = union_indices(t_orig, t_gend, t_pddl)
    for k, pool in (("noun", nouns), ("verb", verbs), ("adj", adjs), ("rel", rels)):
        if len(set(pool)) < len(needs[k]):
            raise ValueError(f"Not enough unique {k}s for union of templates: need {len(needs[k])}, have {len(set(pool))}.")

    rows = []
    seen_triplets = set()

    for i in range(n_samples):
        for attempt in range(80):  # more retries since three templates must match
            assignments = _build_assignment(t_orig + "\n" + t_gend + "\n" + t_pddl, nouns, verbs, adjs, rels)

            filled_orig = _fill_with_assignments(t_orig, assignments).strip()
            filled_gend = _fill_with_assignments(t_gend, assignments).strip()
            filled_pddl = _fill_with_assignments(t_pddl, assignments).strip()

            # Deduplicate across whole triple
            key = (filled_orig, filled_gend, filled_pddl)
            if key not in seen_triplets:
                seen_triplets.add(key)
                mapping = {
                    "noun": assignments["noun"],
                    "verb": assignments["verb"],
                    "adj": assignments["adj"],
                    "rel": assignments["rel"],
                }
                rows.append({
                    "id": i + 1,
                    "mapping": json.dumps(mapping, ensure_ascii=False),
                    "prompt_original": filled_orig,
                    "prompt_generated_domain": filled_gend,
                    "prompt_pddl": filled_pddl,
                })
                break
        else:
            print(f"Stopped early at {len(rows)} (triple) prompts — reached uniqueness limit.")
            break

    # Write CSV
    outp = Path(out_csv_path)
    outp.parent.mkdir(parents=True, exist_ok=True)
    with outp.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["id", "mapping", "prompt_original", "prompt_generated_domain", "prompt_pddl"]
        )
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
import os
from pathlib import Path

# Word lists
NOUN_PATH = "../word_lists/nonce_noun.txt"
VERB_PATH = "../word_lists/nonce_verb.txt"
ADJ_PATH  = "../word_lists/nonce_adj.txt"
REL_PATH  = "../word_lists/nonce_rel.txt"

# Bases
ORIG_BASES = [
    "../prompts/blocksworld/easy",
    "../prompts/blocksworld/medium",
    "../prompts/blocksworld/hard",
    "../prompts/blocksworld/super_hard",
    "../prompts/mystery_blocksworld/easy",
    "../prompts/mystery_blocksworld/medium",
    "../prompts/mystery_blocksworld/hard",
    "../prompts/mystery_blocksworld/super_hard",
]

# Sibling bases that share EXACT filenames by difficulty
GEN_DOMAIN_BASE_BLOCKS = "../prompts/blocksworld_generated_domain"
PDDL_BASE_BLOCKS       = "../prompts/blocksworld_pddl"

GEN_DOMAIN_BASE_MYST   = "../prompts/mystery_blocksworld_generated_domain"
PDDL_BASE_MYST         = "../prompts/mystery_blocksworld_pddl"

# Output roots (split by family)
OUT_BLOCKS = "../outputs/generated_prompts/blocksworld_triples"
OUT_MYST   = "../outputs/generated_prompts/mystery_blocksworld_triples"

N_SAMPLES = 500
SEED = 58

def _family_paths(base_dir: str, difficulty: str, stem: str) -> tuple[Path, Path]:
    """Return paths to generated_domain and pddl siblings given the original family."""
    if "mystery_blocksworld" in base_dir:
        gd = Path(GEN_DOMAIN_BASE_MYST) / difficulty / f"{stem}.txt"
        pd = Path(PDDL_BASE_MYST)       / difficulty / f"{stem}.txt"
    else:
        gd = Path(GEN_DOMAIN_BASE_BLOCKS) / difficulty / f"{stem}.txt"
        pd = Path(PDDL_BASE_BLOCKS)       / difficulty / f"{stem}.txt"
    return gd, pd

for base_dir in ORIG_BASES:
    base_path = Path(base_dir)

    # Output root by family
    out_root = Path(OUT_MYST if "mystery_blocksworld" in base_dir else OUT_BLOCKS)

    for template_path in base_path.glob("*.txt"):
        difficulty = template_path.parent.name
        stem = template_path.stem

        gen_domain_path, pddl_path = _family_paths(base_dir, difficulty, stem)

        if not gen_domain_path.exists() or not pddl_path.exists():
            print(f"⚠️ Skipping (missing sibling): {template_path.name}")
            print(f"   gen_domain exists? {gen_domain_path.exists()}  pddl exists? {pddl_path.exists()}")
            continue

        out_dir = out_root / difficulty
        out_dir.mkdir(parents=True, exist_ok=True)

        out_csv = out_dir / f"{stem}_triples_{N_SAMPLES}.csv"

        print(f"Processing triple:")
        print(f"  original       : {template_path}")
        print(f"  generated_domain: {gen_domain_path}")
        print(f"  pddl           : {pddl_path}")
        print(f"  -> Output CSV  : {out_csv}")

        create_prompts_triple(
            template_path=str(template_path),
            gen_domain_path=str(gen_domain_path),
            pddl_path=str(pddl_path),
            noun_path=NOUN_PATH,
            verb_path=VERB_PATH,
            adj_path=ADJ_PATH,
            rel_path=REL_PATH,
            out_csv_path=str(out_csv),
            n_samples=N_SAMPLES,
            seed=SEED
        )


Processing triple:
  original       : ../prompts/blocksworld/easy/blockworld_easy_actions.txt
  generated_domain: ../prompts/blocksworld_generated_domain/easy/blockworld_easy_actions.txt
  pddl           : ../prompts/blocksworld_pddl/easy/blockworld_easy_actions.txt
  -> Output CSV  : ../outputs/generated_prompts/blocksworld_triples/easy/blockworld_easy_actions_triples_500.csv


KeyError: '1'